# 1.1 · SQL 基础 / SQL Basics

> **课程定位 / Where this fits**
> **Part 1 第 1 课**。Part 0 教了 Python / 数学 / 工程，从这一 Part 起，我们正式进入 **数据 / 算法 / 系统** 的核心内容。
> **Part 1, lesson 1.** Part 0 built up Python / math / engineering; from here on we cover the data / algorithm / system core.
>
> SQL 是数据科学的"母语"——**面试第一关，工作每天都用**。这一节我们用 DuckDB 在 notebook 里直接跑 SQL，把 SELECT / WHERE / ORDER / DISTINCT / CASE / 聚合一次串完。
> SQL is DS's lingua franca. We use DuckDB to run SQL inline in the notebook.

> 📐 **符号约定 / Notation**
> SQL 关键字**全大写**（`SELECT`, `WHERE`, `FROM` ...）—— 不是语法要求，是行业惯例（让查询语句和列名/表名一眼可分）。
> SQL keywords ALL CAPS by convention (not required).

> 💡 **面试相关 / Interview-relevant**
> - "用 SQL 算每个用户的次日留存" ★★★★★（基础题）
> - "找出连续登录 7 天的用户" ★★★★（要用窗口函数 + 1.5 节再讲）
> - "为什么 `WHERE` 不能用聚合函数" ★★★★
> - "`NULL` 和 `0` / 空字符串的区别" ★★★★
>
> Common interview hits: retention, consecutive logins (needs window funcs from 1.5), why aggregates don't work in WHERE, NULL semantics.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 解释**关系型数据库**的核心概念：表、行、列、主键、外键。
   Explain RDBMS basics: tables, rows, columns, primary / foreign keys.
2. 写出 `SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT` 的标准查询。
   Write the canonical `SELECT ... WHERE ... ORDER BY ... LIMIT` query.
3. 熟练用 `DISTINCT` / `LIKE` / `IN` / `BETWEEN` / `IS NULL`。
   Use `DISTINCT` / `LIKE` / `IN` / `BETWEEN` / `IS NULL` fluently.
4. 用 `CASE WHEN` 做"派生列"和"分桶"。
   Use `CASE WHEN` for derived columns / bucketing.
5. 用聚合函数 `COUNT / SUM / AVG / MIN / MAX` 算单一汇总值。
   Use the five basic aggregates (group-by is next lesson).
6. **在 Python 里用 DuckDB 跑 SQL**，无缝衔接 pandas。
   Run SQL in Python via DuckDB, seamlessly with pandas.

---

## 目录 / Table of Contents

1. [关系型数据库速览 / RDBMS Recap](#1)
2. [🎵 数据集：Mini Music Store](#2)
3. [`SELECT` 基础](#3)
4. [`WHERE` 过滤](#4)
5. [`ORDER BY` 排序](#5)
6. [`LIMIT / OFFSET` 分页](#6)
7. [`DISTINCT` 去重](#7)
8. [`LIKE` / `IN` / `BETWEEN`](#8)
9. [⚠ NULL 处理](#9)
10. [`CASE WHEN` 派生列](#10)
11. [基础聚合 / Basic Aggregates](#11)
12. [Python ↔ SQL 集成](#12)
13. [实战：业务问题 10 连击 / Hands-on](#13)
14. [小结 / Summary](#14)


<a id="1"></a>
## 1. 关系型数据库速览 / RDBMS Recap

### 1.1 核心概念 / Core concepts

| 概念 / Concept | 含义 / Meaning |
|---|---|
| **表 / Table** | 二维结构（行 + 列）/ 2D structure |
| **行 / Row** | 一条记录 / one record |
| **列 / Column** | 一个属性 / one attribute |
| **主键 / Primary key (PK)** | 唯一标识一行（如 `customer_id`）/ uniquely IDs a row |
| **外键 / Foreign key (FK)** | 指向另一张表的主键 / points to another table's PK |
| **Schema** | 表结构 + 约束 / table structure + constraints |

### 1.2 数据库 vs DataFrame

| 维度 / Aspect | RDBMS | Pandas |
|---|---|---|
| 持久化 / Persistence | ✅ 磁盘 | ❌ 内存 |
| 并发 / Concurrency | ✅ 多用户事务 | ❌ 单进程 |
| 数据量 / Scale | TB+ | RAM 限制 |
| 类型严格 / Strict types | ✅ schema 锁住 | ⚠ 可能改 |

> 💡 **生产架构**：原始数据在 **DB**（Postgres / MySQL / Snowflake），分析师**用 SQL 拉子集**到 pandas/Polars 里做精细处理。
> Production: raw data in DB, pull subsets via SQL to pandas/Polars for fine work.

### 1.3 SQL 方言 / SQL dialects

- **PostgreSQL** —— 开源王者，DS 最常见
- **MySQL** —— 老牌，互联网公司多
- **SQLite** —— 单文件嵌入式
- **DuckDB** —— **新王**：本地分析 OLAP 神器，本节用它
- **Snowflake / BigQuery / Redshift** —— 云数仓三巨头
- **SQL Server (T-SQL)** —— 微软系


In [ ]:
import duckdb
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

print(f"duckdb : {duckdb.__version__}")
print(f"pandas : {pd.__version__}")


<a id="2"></a>
## 2. 🎵 数据集：Mini Music Store / Dataset

> 我们造一个"迷你音乐商店"，灵感来自经典的 **Chinook** sample DB。**5 张表**互相关联。
> A toy music store inspired by the classic **Chinook**. Five interrelated tables.
>
> | 表 / Table | 字段 / Fields | 主键 | 外键 |
> |---|---|---|---|
> | `artist`   | `artist_id, name, country` | `artist_id` | — |
> | `album`    | `album_id, title, artist_id, year` | `album_id` | `artist_id` |
> | `track`    | `track_id, name, album_id, genre, seconds, price` | `track_id` | `album_id` |
> | `customer` | `customer_id, name, country, email` | `customer_id` | — |
> | `invoice`  | `invoice_id, customer_id, track_id, invoice_date, quantity` | `invoice_id` | 2 个 |
>
> **规模 / Size**: 6 artists × 8 albums × 22 tracks × 6 customers × 20 invoices.


In [ ]:
conn = duckdb.connect()        # in-memory DB

conn.sql("""
CREATE TABLE artist (
    artist_id INTEGER PRIMARY KEY,
    name      VARCHAR,
    country   VARCHAR
);
INSERT INTO artist VALUES
    (1, 'The Beatles',     'UK'),
    (2, 'Pink Floyd',      'UK'),
    (3, 'Miles Davis',     'US'),
    (4, 'Daft Punk',       'FR'),
    (5, 'Radiohead',       'UK'),
    (6, 'Anonymous Artist', NULL);     -- 故意留个 NULL country / intentional NULL
""")
print(conn.sql("SELECT * FROM artist;").df())


In [ ]:
conn.sql("""
CREATE TABLE album (
    album_id  INTEGER PRIMARY KEY,
    title     VARCHAR,
    artist_id INTEGER,
    year      INTEGER
);
INSERT INTO album VALUES
    (1, 'Abbey Road',                    1, 1969),
    (2, 'The Dark Side of the Moon',     2, 1973),
    (3, 'The Wall',                      2, 1979),
    (4, 'Kind of Blue',                  3, 1959),
    (5, 'Discovery',                     4, 2001),
    (6, 'Random Access Memories',        4, 2013),
    (7, 'OK Computer',                   5, 1997),
    (8, 'Demos (unreleased)',            6, 2024);
""")
print(conn.sql("SELECT * FROM album;").df())


In [ ]:
conn.sql("""
CREATE TABLE track (
    track_id  INTEGER PRIMARY KEY,
    name      VARCHAR,
    album_id  INTEGER,
    genre     VARCHAR,
    seconds   INTEGER,
    price     DECIMAL(4, 2)
);
INSERT INTO track VALUES
    (1,  'Come Together',           1, 'Rock',       259, 0.99),
    (2,  'Something',                1, 'Rock',       182, 0.99),
    (3,  'Here Comes the Sun',       1, 'Rock',       185, 0.99),
    (4,  'Time',                     2, 'Rock',       413, 1.29),
    (5,  'Money',                    2, 'Rock',       382, 1.29),
    (6,  'Us and Them',              2, 'Rock',       460, 1.29),
    (7,  'Another Brick in the Wall',3, 'Rock',       239, 1.29),
    (8,  'Comfortably Numb',         3, 'Rock',       382, 1.29),
    (9,  'So What',                  4, 'Jazz',       545, 1.49),
    (10, 'Freddie Freeloader',       4, 'Jazz',       586, 1.49),
    (11, 'Blue in Green',            4, 'Jazz',       337, 1.49),
    (12, 'One More Time',            5, 'Electronic', 320, 1.29),
    (13, 'Aerodynamic',              5, 'Electronic', 213, 1.29),
    (14, 'Digital Love',             5, 'Electronic', 301, 1.29),
    (15, 'Get Lucky',                6, 'Electronic', 369, 1.29),
    (16, 'Instant Crush',            6, 'Electronic', 337, 1.29),
    (17, 'Lose Yourself to Dance',   6, 'Electronic', 353, 1.29),
    (18, 'Paranoid Android',         7, 'Rock',       384, 1.29),
    (19, 'Karma Police',             7, 'Rock',       261, 1.29),
    (20, 'No Surprises',             7, 'Rock',       228, 1.29),
    (21, 'Untitled Demo 1',          8, 'Rock',       180, 0.50),
    (22, 'Untitled Demo 2',          8, 'Rock',       195, 0.50);
""")
print(f"track total: {conn.sql('SELECT COUNT(*) FROM track;').fetchone()[0]}")


In [ ]:
conn.sql("""
CREATE TABLE customer (
    customer_id INTEGER PRIMARY KEY,
    name        VARCHAR,
    country     VARCHAR,
    email       VARCHAR
);
INSERT INTO customer VALUES
    (1, 'Alice Chen',    'US', 'alice@example.com'),
    (2, 'Bob Smith',     'UK', 'BOB@example.com'),
    (3, 'Charlie Davis', 'US', 'charlie@example.com'),
    (4, 'Diana Park',    'DE', 'diana@example.com'),
    (5, 'Ethan Miller',  'US', 'ethan@example.com'),
    (6, 'Fiona Wong',    'JP', NULL);     -- NULL email
""")

conn.sql("""
CREATE TABLE invoice (
    invoice_id   INTEGER PRIMARY KEY,
    customer_id  INTEGER,
    track_id     INTEGER,
    invoice_date DATE,
    quantity     INTEGER
);
INSERT INTO invoice VALUES
    (1,  1, 1,  DATE '2026-01-05', 1),
    (2,  1, 9,  DATE '2026-01-05', 2),
    (3,  2, 4,  DATE '2026-01-10', 1),
    (4,  2, 18, DATE '2026-01-10', 1),
    (5,  3, 5,  DATE '2026-02-12', 1),
    (6,  3, 6,  DATE '2026-02-12', 1),
    (7,  3, 7,  DATE '2026-02-12', 3),
    (8,  4, 15, DATE '2026-02-20', 1),
    (9,  4, 12, DATE '2026-02-20', 1),
    (10, 4, 13, DATE '2026-02-20', 1),
    (11, 5, 9,  DATE '2026-03-01', 1),
    (12, 5, 10, DATE '2026-03-01', 1),
    (13, 5, 11, DATE '2026-03-01', 1),
    (14, 5, 4,  DATE '2026-03-05', 2),
    (15, 1, 18, DATE '2026-03-15', 1),
    (16, 1, 19, DATE '2026-03-15', 1),
    (17, 2, 15, DATE '2026-04-01', 1),
    (18, 3, 14, DATE '2026-04-10', 2),
    (19, 4, 8,  DATE '2026-05-02', 1),
    (20, 5, 1,  DATE '2026-05-20', 1);
""")
print(f"customer total: {conn.sql('SELECT COUNT(*) FROM customer;').fetchone()[0]}")
print(f"invoice  total: {conn.sql('SELECT COUNT(*) FROM invoice;').fetchone()[0]}")


<a id="3"></a>
## 3. `SELECT` 基础

最小可用 SELECT:
```sql
SELECT <columns> FROM <table>;
```


In [ ]:
conn.sql("SELECT * FROM artist;").df()


In [ ]:
conn.sql("""
    SELECT name, country
    FROM artist;
""").df()


In [ ]:
# 列别名 / Column aliases
conn.sql("""
    SELECT
        name    AS artist_name,
        country AS origin
    FROM artist;
""").df()


**SELECT 表达式**: 不只能取列，还能**算列**。
**SELECT can compute** — not just project columns.


In [ ]:
conn.sql("""
    SELECT
        name,
        seconds,
        ROUND(seconds / 60.0, 2) AS minutes,
        price,
        price * 1.10              AS price_after_tax
    FROM track
    LIMIT 5;
""").df()


<a id="4"></a>
## 4. `WHERE` 过滤 / Filtering Rows

`WHERE` 在 `FROM` 之后、`SELECT` 之前**逻辑执行**——筛选哪些**行**进来。

```sql
SELECT ... FROM ... WHERE <condition>;
```

条件运算：
- 比较：`=`, `!=` / `<>`, `<`, `>`, `<=`, `>=`
- 逻辑：`AND`, `OR`, `NOT`
- 模式：`LIKE`, `IN`, `BETWEEN`, `IS NULL`


In [ ]:
conn.sql("""
    SELECT title AS name, year
    FROM album
    WHERE year < 1980;
""").df()


In [ ]:
# 多条件：括号清晰 / Combine; parens for clarity
conn.sql("""
    SELECT title AS name, year, artist_id
    FROM album
    WHERE (year BETWEEN 1990 AND 2010)
       OR (artist_id = 1);
""").df()


> ⚠ **常踩的坑 / Common gotchas**
> - `WHERE` 里**不能直接用聚合函数** —— 那是 `HAVING` 的事
> - 字符串等于用 **`=`** 不是 `==`
> - 不等于是 **`<>`** 或 `!=`（前者更标准）
> - 字符串比较通常**区分大小写**（DuckDB / Postgres）
>
> - WHERE cannot use aggregates (use HAVING)
> - String comparisons usually case-sensitive in DuckDB / Postgres


<a id="5"></a>
## 5. `ORDER BY` 排序

```sql
SELECT ... ORDER BY col1 [ASC|DESC], col2 [ASC|DESC];
```
- 默认 `ASC`
- 多列：先按 col1，col1 相同的按 col2


In [ ]:
# 价格降序，时长升序 / Price desc, then duration asc
conn.sql("""
    SELECT name, price, seconds
    FROM track
    ORDER BY price DESC, seconds ASC
    LIMIT 8;
""").df()


**小技巧**: 可以用列序号或别名排序：
Pro tip: `ORDER BY` accepts column numbers or aliases.

```sql
SELECT name, price * 1.1 AS taxed FROM track ORDER BY taxed DESC;
SELECT name, year FROM album ORDER BY 2 DESC;   -- 按第 2 列
```


<a id="6"></a>
## 6. `LIMIT / OFFSET` 分页

```sql
SELECT ... LIMIT 10;            -- 前 10 行
SELECT ... LIMIT 10 OFFSET 20;  -- 跳过 20 行再取 10 行
```

**典型应用**：网页分页（第 3 页每页 10 条 → `OFFSET 20 LIMIT 10`）。


In [ ]:
conn.sql("""
    SELECT name, price
    FROM track
    ORDER BY price DESC
    LIMIT 3;
""").df()


In [ ]:
# 第 4-6 名 / Ranks 4-6
conn.sql("""
    SELECT name, price
    FROM track
    ORDER BY price DESC
    LIMIT 3 OFFSET 3;
""").df()


<a id="7"></a>
## 7. `DISTINCT` 去重


In [ ]:
conn.sql("""
    SELECT DISTINCT genre
    FROM track;
""").df()


In [ ]:
# DISTINCT 多列：唯一组合 / Distinct on multiple columns
conn.sql("""
    SELECT DISTINCT
        country,
        email IS NULL AS no_email
    FROM customer;
""").df()


<a id="8"></a>
## 8. `LIKE` / `IN` / `BETWEEN`

### 8.1 `LIKE` 模式匹配

| 通配符 | 含义 |
|---|---|
| `%` | 0 或多个字符 |
| `_` | 恰 1 个字符 |


In [ ]:
# 以 "The" 开头的艺术家 / Starts with "The"
conn.sql("""
    SELECT name FROM artist
    WHERE name LIKE 'The%';
""").df()


In [ ]:
# 不区分大小写：用 ILIKE (DuckDB / Postgres)
conn.sql("""
    SELECT name FROM track
    WHERE name ILIKE '%love%';
""").df()


### 8.2 `IN` 集合 / Set membership


In [ ]:
# 比一堆 OR 紧凑 / Compact alternative to OR-chain
conn.sql("""
    SELECT name, country
    FROM customer
    WHERE country IN ('US', 'UK', 'DE');
""").df()


### 8.3 `BETWEEN` 范围 / Range

`BETWEEN a AND b` 是 **inclusive** 的（两端都含）。


In [ ]:
# 90 年代专辑 / Albums from the 90s
conn.sql("""
    SELECT title, year
    FROM album
    WHERE year BETWEEN 1990 AND 1999;
""").df()


<a id="9"></a>
## 9. ⚠ NULL 处理

**SQL 里 `NULL` 不是 0，不是空字符串，是"不知道"**。
**NULL is not 0, not '', not False — it's "unknown".**

### 关键规则
1. `NULL = NULL` 不是 TRUE，是 `NULL`！必须 `IS NULL` / `IS NOT NULL`
2. 算术：`NULL + 1 = NULL`，任何含 NULL 的运算都返回 NULL
3. 聚合：`COUNT(col)` **忽略 NULL**；`COUNT(*)` 不忽略
4. `WHERE col = X` **不返回 col 为 NULL 的行**

### 面试题

> 反选所有不等于 'X' 的行——为什么 NULL 行会丢？必须 `WHERE col <> 'X' OR col IS NULL`。
> Why does "negate equals X" silently drop NULL rows? Need explicit `IS NULL`.


In [ ]:
# 没填 email 的客户 / Missing emails
print(conn.sql("""
    SELECT name, email
    FROM customer
    WHERE email IS NULL;
""").df())

# 错误写法：什么也不返回 / Wrong — silent bug
print("\n--- WHERE email = NULL ---")
print(conn.sql("""
    SELECT name, email
    FROM customer
    WHERE email = NULL;
""").df())


### `COALESCE` / `NULLIF` / `IFNULL`

| 函数 | 作用 |
|---|---|
| `COALESCE(a, b, c, ...)` | 第一个非 NULL 值 |
| `NULLIF(a, b)` | `a = b` 时返回 NULL，否则返回 `a` |
| `IFNULL(a, b)` | `COALESCE(a, b)` 的 MySQL 别名 |


In [ ]:
# COALESCE: 给缺失值一个占位 / Default fallback
conn.sql("""
    SELECT
        name,
        COALESCE(email, '(no email)') AS contact_email
    FROM customer;
""").df()


<a id="10"></a>
## 10. `CASE WHEN` —— SQL 的 if-else

```sql
CASE
    WHEN <cond1> THEN <value1>
    WHEN <cond2> THEN <value2>
    ELSE <default>
END
```

**派生列 / 分桶必备**。无 `ELSE` 时不匹配的行得到 `NULL`。


In [ ]:
# 按时长分桶 / Bucket by duration
conn.sql("""
    SELECT
        name,
        seconds,
        CASE
            WHEN seconds < 200 THEN 'short'
            WHEN seconds < 350 THEN 'medium'
            WHEN seconds < 500 THEN 'long'
            ELSE 'epic'
        END AS length_bucket
    FROM track
    ORDER BY seconds
    LIMIT 10;
""").df()


In [ ]:
# 条件计数 trick / Conditional counting
conn.sql("""
    SELECT
        SUM(CASE WHEN price >= 1.29 THEN 1 ELSE 0 END) AS expensive_count,
        SUM(CASE WHEN price < 1.00  THEN 1 ELSE 0 END) AS cheap_count,
        COUNT(*) AS total
    FROM track;
""").df()


`SUM(CASE WHEN ...)` 是"条件计数"在没有 `FILTER` 子句的方言里的标准做法。
The `SUM(CASE WHEN ...)` pattern is the canonical conditional-count idiom in dialects without `FILTER`.


<a id="11"></a>
## 11. 基础聚合 / Basic Aggregates

| 函数 | 作用 |
|---|---|
| `COUNT(*)` | 行数（含 NULL） |
| `COUNT(col)` | 非 NULL 行数 |
| `COUNT(DISTINCT col)` | 去重计数 |
| `SUM(col)` | 求和 |
| `AVG(col)` | 均值 |
| `MIN(col)`, `MAX(col)` | 最小、最大 |

**本节只看"单组聚合"**。**按组聚合** = `GROUP BY` 是下一节的主角。
**Single-row aggregates only here**; multi-group = `GROUP BY` in 1.3.


In [ ]:
conn.sql("""
    SELECT
        COUNT(*)                        AS total_invoices,
        COUNT(DISTINCT customer_id)     AS unique_customers,
        SUM(quantity)                   AS total_units_sold,
        ROUND(AVG(quantity), 2)         AS avg_qty_per_invoice,
        MIN(invoice_date)               AS first_date,
        MAX(invoice_date)               AS last_date
    FROM invoice;
""").df()


In [ ]:
# COUNT(*) vs COUNT(col)：NULL 区别 / Difference around NULLs
conn.sql("""
    SELECT
        COUNT(*)                AS row_count,
        COUNT(email)            AS non_null_emails,
        COUNT(*) - COUNT(email) AS missing_emails
    FROM customer;
""").df()


<a id="12"></a>
## 12. Python ↔ SQL 集成

DuckDB 的 killer feature：**直接查 pandas DataFrame**，无须 import 到 DB。
DuckDB's killer feature: query DataFrames directly.


In [ ]:
# 1) DuckDB → pandas
df_albums = conn.sql("SELECT * FROM album").df()
print(type(df_albums))
print(df_albums.head(3))


In [ ]:
# 2) pandas → DuckDB —— 直接在 SQL 里引用 Python 变量！
my_df = pd.DataFrame({
    "id":    [1, 2, 3],
    "name":  ["A", "B", "C"],
    "score": [88, 95, 72],
})

result = conn.sql("""
    SELECT name, score
    FROM my_df                  -- ← Python 变量名 / Python variable
    WHERE score > 80
    ORDER BY score DESC;
""").df()
print(result)


In [ ]:
# 3) SQL + pandas 混合 / Hybrid analysis
tpa = conn.sql("""
    SELECT album_id, COUNT(*) AS n_tracks
    FROM track
    GROUP BY album_id;
""").df()

album_df = conn.sql("SELECT * FROM album").df()
merged = album_df.merge(tpa, on="album_id").sort_values("n_tracks", ascending=False)
print(merged[["title", "year", "n_tracks"]])


**这就是现代 DS 工作流**：
- **SQL 做"重活"**（filter / join / group）在大量数据上
- **Pandas / Polars 做"精细活"**（建模特征、可视化）

The modern DS workflow: SQL does the heavy lifting on large data; pandas/Polars finer modeling and viz.


<a id="13"></a>
## 13. 实战：业务问题 10 连击 / 10 Business Questions

把这一节的工具串成"音乐商店分析师" mini case。**每题先想 SQL 怎么写，再看答案**。


In [ ]:
# Q1: 共有多少张专辑？/ How many albums?
conn.sql("SELECT COUNT(*) AS n_albums FROM album").df()


In [ ]:
# Q2: 所有 Rock 歌曲，按时长降序 / All rock tracks, longest first
conn.sql("""
    SELECT name, seconds, ROUND(seconds/60.0, 2) AS minutes
    FROM track
    WHERE genre = 'Rock'
    ORDER BY seconds DESC
    LIMIT 5;
""").df()


In [ ]:
# Q3: 不重复的"类型 × 价格" / Unique (genre, price) combos
conn.sql("""
    SELECT DISTINCT genre, price
    FROM track
    ORDER BY genre, price;
""").df()


In [ ]:
# Q4: 来自 UK 的艺术家 / UK artists
conn.sql("""
    SELECT name, country
    FROM artist
    WHERE country = 'UK';
""").df()


In [ ]:
# Q5: 没填 country 的艺术家 / Artists with missing country
conn.sql("""
    SELECT artist_id, name
    FROM artist
    WHERE country IS NULL;
""").df()


In [ ]:
# Q6: 给所有歌曲分价格档 / Label every track by price tier
conn.sql("""
    SELECT
        name,
        price,
        CASE
            WHEN price < 1.00 THEN 'cheap'
            WHEN price < 1.30 THEN 'standard'
            ELSE 'premium'
        END AS price_tier
    FROM track
    ORDER BY price, name
    LIMIT 8;
""").df()


In [ ]:
# Q7: 平均歌曲时长 / Avg track length
conn.sql("""
    SELECT
        ROUND(AVG(seconds), 1)        AS avg_seconds,
        ROUND(AVG(seconds) / 60.0, 2) AS avg_minutes
    FROM track;
""").df()


In [ ]:
# Q8: 邮箱缺失率 / Email missing rate
conn.sql("""
    SELECT
        COUNT(*)                                                            AS total,
        COUNT(email)                                                        AS with_email,
        COUNT(*) - COUNT(email)                                              AS missing,
        ROUND((COUNT(*) - COUNT(email)) * 100.0 / COUNT(*), 1)               AS missing_pct
    FROM customer;
""").df()


In [ ]:
# Q9: 包含 "love" 的歌（不分大小写）/ Tracks containing "love"
conn.sql("""
    SELECT name
    FROM track
    WHERE name ILIKE '%love%';
""").df()


In [ ]:
# Q10: 2026 Q1 (Jan-Mar) 发票数 / Invoices in Q1 2026
conn.sql("""
    SELECT COUNT(*) AS q1_invoices
    FROM invoice
    WHERE invoice_date BETWEEN DATE '2026-01-01' AND DATE '2026-03-31';
""").df()


<a id="14"></a>
## 14. 小结 / Summary

### 概念地图 / Concept map

```
SELECT <expr>
  FROM <table>
  WHERE <row filter>           ← 过滤行（不能用聚合）
  ORDER BY <col> [ASC|DESC]    ← 排序
  LIMIT n OFFSET k             ← 分页

工具箱:
  - DISTINCT     去重
  - LIKE / ILIKE 模式匹配
  - IN / BETWEEN 范围
  - IS NULL      NULL 处理 ⚠
  - CASE WHEN    SQL 的 if-else
  - COALESCE     NULL 填充
  - COUNT/SUM/AVG/MIN/MAX  单组聚合
```

### 💡 一句话精华 / One-line gist

| Pattern | Why |
|---|---|
| `SELECT col` 不是 `SELECT *` | 性能 + 显式 |
| `WHERE` 不能用聚合 | 那是 `HAVING` 的事 |
| `IS NULL`，**不是** `= NULL` | NULL 不可比较 |
| `LIKE 'abc%'` ≠ `LIKE '%abc%'` | 后者用不上前缀索引 |
| `COUNT(*) ≠ COUNT(col)` 当 col 有 NULL | 面试坑 |
| 字符串等于：`'a' = 'A'` 通常 **FALSE** | 用 `ILIKE` 或 `LOWER()` |

### 💡 工业速查

```sql
-- 抽样调试 / Quick sample for debugging
SELECT * FROM big_table LIMIT 5;

-- 看表 schema
DESCRIBE my_table;                  -- DuckDB / MySQL

-- 字符串拼接 / String concat
SELECT first_name || ' ' || last_name FROM users;   -- ANSI / Postgres / DuckDB
SELECT CONCAT(first_name, ' ', last_name) FROM users;   -- MySQL

-- 当前日期 / Current date
SELECT CURRENT_DATE;
SELECT NOW();

-- 类型转换 / Cast
SELECT CAST('2026-01-01' AS DATE);
SELECT '2026-01-01'::DATE;          -- Postgres / DuckDB 简写
```

### 💡 面试速查 / Interview must-knows

1. **WHERE vs HAVING**：WHERE 在 GROUP BY 之前；HAVING 在之后（专为聚合）
2. **NULL 三件事**：用 `IS NULL`、`COUNT(col)` 忽略 NULL、`COALESCE` 兜底
3. **`SELECT *` 是反 pattern**：实际工作只取需要的列
4. **`LIKE '%xxx'` 用不上 B-tree 索引**（1.7 节讲索引时回顾）
5. **写 SQL 的标准缩进**：`SELECT`、`FROM`、`WHERE` 各起一行

### 下一节预告 / Next up

**Part 1.2 · 多表 JOIN** —— `INNER` / `LEFT` / `RIGHT` / `FULL` / `CROSS` / `SELF` JOIN 一次讲透，含面试题反例。
**Part 1.2 · JOINs** — Inner / Left / Right / Full / Cross / Self with interview gotchas.
